# Symbolic Eligibility Rule Encoding

This notebook converts selected ClinicalTrials.gov eligibility criteria into structured symbolic rules and evaluates synthetic osteosarcoma patients using three-valued logic: MET, NOT_MET, and UNKNOWN.

In [2]:
import json
from pathlib import Path
import pandas as pd

In [3]:
PROJECT_ROOT = Path("..")

TRIALS_DIR = PROJECT_ROOT / "data" / "trials"
PATIENTS_DIR = PROJECT_ROOT / "data" / "synthetic_patients"
RULES_DIR = PROJECT_ROOT / "data" / "rules"

RULES_DIR.mkdir(parents=True, exist_ok=True)

print("Trials directory:", TRIALS_DIR.resolve())
print("Patients directory:", PATIENTS_DIR.resolve())
print("Rules directory:", RULES_DIR.resolve())

Trials directory: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\trials
Patients directory: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\synthetic_patients
Rules directory: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rules


In [4]:
five_trials = pd.read_csv(
    TRIALS_DIR / "five_proposed_trials.csv"
)

print("Selected trials:", len(five_trials))

five_trials[
    [
        "NCT Number",
        "Title",
        "Minimum Age",
        "Maximum Age"
    ]
]

Selected trials: 5


,NCT Number,Title,Minimum Age,Maximum Age
0,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,7 Years,22 Years
1,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,1 Year,30 Years
2,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,2 Years,25 Years
3,NCT06235125,Safety and Feasibility of Intraoperative Visua...,6 Years,17 Years
4,NCT02013336,Phase 1 Dose-escalating Study of MM-398 (Irino...,12 Months,20 Years


In [5]:
with open(
    PATIENTS_DIR / "synthetic_patients.json",
    "r",
    encoding="utf-8"
) as f:
    synthetic_patients = json.load(f)

print("Synthetic patients loaded:", len(synthetic_patients))

for patient in synthetic_patients:
    print(
        patient["patient_id"],
        "-",
        patient["age_years"],
        "-",
        patient["diagnosis"]["disease_status"]
    )

Synthetic patients loaded: 8
P001 - 15 - newly_diagnosed
P002 - 16 - newly_diagnosed
P003 - 17 - relapsed
P004 - 14 - refractory
P005 - 12 - refractory
P006 - 13 - metastatic
P007 - 10 - recurrent
P008 - 19 - relapsed


In [6]:
trial_07560826 = five_trials[
    five_trials["NCT Number"] == "NCT07560826"
].iloc[0]

print("Trial:")
print(trial_07560826["Title"])

print("\nEligibility Criteria:")
print(trial_07560826["Eligibility Text"])

Trial:
The CATSINDO Trial - Clinical and Translational Study in Newly Diagnosed Osteosarcoma

Eligibility Criteria:
Pre-Enrollment Criteria:

1. Suspected diagnosis of high-grade osteosarcoma based on clinical and radiographic findings.
2. Informed consent (and assent, if applicable) obtained, per institutional guidelines.
3. Participants must be ≤ 22 years of age at the time of consent.

Inclusion Criteria:

1. Participants with localized or metastatic high-grade osteosarcoma.
2. Participants must be ≤ 22 years of age at the time of consent.
3. Participants must have a body surface area of greater than or equal to 0.8 m2.
4. Participants receiving or planning to receive induction/neoadjuvant MAP chemotherapy with HD-MTX given at the standard dose of 12 gm/m2 (maximum 20 gm).

   NOTE: Participants may be participating on other clinical studies such as the COG trial AOST2032 or any other clinical trial as long as their treatment includes MAP chemotherapy with the standard HD-MTX dose o

## Symbolic Rule Representation

Each eligibility criterion is represented as a structured rule containing:
- a unique rule ID
- the patient field to inspect
- an operator
- the expected value or threshold
- the original criterion text
- whether the rule is an inclusion or exclusion criterion

The rule engine will later evaluate each rule as MET, NOT_MET, or UNKNOWN.

In [7]:
def make_rule(
    rule_id,
    field,
    operator,
    value,
    criterion_type,
    description
):
    return {
        "rule_id": rule_id,
        "field": field,
        "operator": operator,
        "value": value,
        "criterion_type": criterion_type,
        "description": description
    }

## NCT07560826 — CATSINDO Symbolic Rules

These rules encode the machine-checkable eligibility requirements from the ClinicalTrials.gov eligibility text.

In [8]:
rules_NCT07560826 = [

    make_rule(
        "NCT07560826_R01",
        "age_years",
        "<=",
        22,
        "inclusion",
        "Participant must be 22 years of age or younger."
    ),

    make_rule(
        "NCT07560826_R02",
        "diagnosis.cancer_type",
        "==",
        "osteosarcoma",
        "inclusion",
        "Participant must have osteosarcoma."
    ),

    make_rule(
        "NCT07560826_R03",
        "diagnosis.grade",
        "==",
        "high-grade",
        "inclusion",
        "Participant must have high-grade osteosarcoma."
    ),

    make_rule(
        "NCT07560826_R04",
        "diagnosis.disease_extent",
        "in",
        ["localized", "metastatic"],
        "inclusion",
        "Localized or metastatic high-grade osteosarcoma is permitted."
    ),

    make_rule(
        "NCT07560826_R05",
        "anthropometrics.body_surface_area_m2",
        ">=",
        0.8,
        "inclusion",
        "Body surface area must be at least 0.8 m2."
    ),

    make_rule(
        "NCT07560826_R06",
        "current_treatment.map_chemotherapy",
        "==",
        True,
        "inclusion",
        "Participant must be receiving or planning induction/neoadjuvant MAP chemotherapy."
    ),

    make_rule(
        "NCT07560826_R07",
        "current_treatment.methotrexate",
        "==",
        True,
        "inclusion",
        "MAP treatment must include high-dose methotrexate."
    ),

    make_rule(
        "NCT07560826_R08",
        "current_treatment.methotrexate_dose_g_m2",
        "==",
        12,
        "inclusion",
        "High-dose methotrexate is given at the standard dose of 12 g/m2."
    ),

    make_rule(
        "NCT07560826_R09",
        "laboratory_values.bilirubin_uln_ratio",
        "<=",
        1.5,
        "inclusion",
        "Total bilirubin must be no more than 1.5 times the upper limit of normal."
    ),

    make_rule(
        "NCT07560826_R10",
        "laboratory_values.alt_u_l",
        "<=",
        135,
        "inclusion",
        "ALT must be no more than 135 U/L."
    ),

    make_rule(
        "NCT07560826_R11",
        "laboratory_values.gfr_ml_min_1_73m2",
        ">=",
        70,
        "inclusion",
        "GFR or creatinine clearance must be at least 70 mL/min/1.73m2."
    ),

    make_rule(
        "NCT07560826_R12",
        "laboratory_values.anc_per_ul",
        ">=",
        1000,
        "inclusion",
        "Absolute neutrophil count must be at least 1000/uL."
    ),

    make_rule(
        "NCT07560826_R13",
        "laboratory_values.platelets_per_ul",
        ">=",
        100000,
        "inclusion",
        "Platelet count must be at least 100000/uL."
    ),

    make_rule(
        "NCT07560826_R14",
        "laboratory_values.hemoglobin_g_dl",
        ">=",
        8.0,
        "inclusion",
        "Hemoglobin must be at least 8.0 g/dL."
    ),

    make_rule(
        "NCT07560826_R15",
        "cardiac.lvef_percent",
        ">=",
        50,
        "inclusion",
        "Left ventricular ejection fraction must be at least 50 percent."
    ),

    make_rule(
        "NCT07560826_R16",
        "current_treatment.second_cycle_started",
        "==",
        False,
        "inclusion",
        "Participant must enroll before the second cycle of chemotherapy."
    ),

    make_rule(
        "NCT07560826_R17",
        "other_clinical_facts.pregnant",
        "==",
        False,
        "exclusion",
        "Pregnant participants are excluded."
    ),

    make_rule(
        "NCT07560826_R18",
        "other_clinical_facts.breastfeeding",
        "==",
        False,
        "exclusion",
        "Breastfeeding participants are excluded."
    )
]

In [9]:
print("Number of encoded rules:", len(rules_NCT07560826))

pd.DataFrame(rules_NCT07560826)

Number of encoded rules: 18


,rule_id,field,operator,value,criterion_type,description
0,NCT07560826_R01,age_years,<=,22,inclusion,Participant must be 22 years of age or younger.
1,NCT07560826_R02,diagnosis.cancer_type,==,osteosarcoma,inclusion,Participant must have osteosarcoma.
2,NCT07560826_R03,diagnosis.grade,==,high-grade,inclusion,Participant must have high-grade osteosarcoma.
3,NCT07560826_R04,diagnosis.disease_extent,in,"[localized, metastatic]",inclusion,Localized or metastatic high-grade osteosarcom...
4,NCT07560826_R05,anthropometrics.body_surface_area_m2,>=,0.8,inclusion,Body surface area must be at least 0.8 m2.
5,NCT07560826_R06,current_treatment.map_chemotherapy,==,True,inclusion,Participant must be receiving or planning indu...
6,NCT07560826_R07,current_treatment.methotrexate,==,True,inclusion,MAP treatment must include high-dose methotrex...
7,NCT07560826_R08,current_treatment.methotrexate_dose_g_m2,==,12,inclusion,High-dose methotrexate is given at the standar...
8,NCT07560826_R09,laboratory_values.bilirubin_uln_ratio,<=,1.5,inclusion,Total bilirubin must be no more than 1.5 times...
9,NCT07560826_R10,laboratory_values.alt_u_l,<=,135,inclusion,ALT must be no more than 135 U/L.


In [10]:
def get_nested_value(data, field_path):
    keys = field_path.split(".")
    value = data

    for key in keys:
        if not isinstance(value, dict) or key not in value:
            return None

        value = value[key]

    return value

In [11]:
print(
    get_nested_value(
        synthetic_patients[0],
        "age_years"
    )
)

print(
    get_nested_value(
        synthetic_patients[0],
        "laboratory_values.anc_per_ul"
    )
)

print(
    get_nested_value(
        synthetic_patients[0],
        "cardiac.lvef_percent"
    )
)

15
1850
61


## Three-Valued Symbolic Rule Evaluation

Each rule is evaluated using three-valued logic:

- MET: the patient fact satisfies the eligibility rule
- NOT_MET: the patient fact contradicts the eligibility rule
- UNKNOWN: the required patient fact is missing

In [12]:
def evaluate_rule(patient, rule):
    operator = rule["operator"]
    expected_value = rule["value"]

    if operator == "count_true_gte":
        fields = rule["field"]

        values = [
            get_nested_value(patient, field)
            for field in fields
        ]

        true_count = sum(
            value is True
            for value in values
        )

        unknown_count = sum(
            value is None
            for value in values
        )

        if true_count >= expected_value:
            return "MET"

        if true_count + unknown_count >= expected_value:
            return "UNKNOWN"

        return "NOT_MET"

    patient_value = get_nested_value(
        patient,
        rule["field"]
    )

    if patient_value is None:
        return "UNKNOWN"

    try:
        if operator == "==":
            result = patient_value == expected_value

        elif operator == "!=":
            result = patient_value != expected_value

        elif operator == ">=":
            result = patient_value >= expected_value

        elif operator == "<=":
            result = patient_value <= expected_value

        elif operator == ">":
            result = patient_value > expected_value

        elif operator == "<":
            result = patient_value < expected_value

        elif operator == "in":
            result = patient_value in expected_value

        else:
            raise ValueError(
                f"Unsupported operator: {operator}"
            )

    except (TypeError, ValueError):
        return "UNKNOWN"

    return "MET" if result else "NOT_MET"

In [13]:
patients_by_id = {
    patient["patient_id"]: patient
    for patient in synthetic_patients
}

P001 = patients_by_id["P001"]
P002 = patients_by_id["P002"]
P003 = patients_by_id["P003"]
P004 = patients_by_id["P004"]
P005 = patients_by_id["P005"]
P006 = patients_by_id["P006"]
P007 = patients_by_id["P007"]
P008 = patients_by_id["P008"]

print("Patient references created successfully.")

Patient references created successfully.


In [14]:
anc_rule = next(
    rule
    for rule in rules_NCT07560826
    if rule["rule_id"] == "NCT07560826_R12"
)

print(
    "P001 ANC:",
    get_nested_value(P001, anc_rule["field"]),
    "→",
    evaluate_rule(P001, anc_rule)
)

print(
    "P002 ANC:",
    get_nested_value(P002, anc_rule["field"]),
    "→",
    evaluate_rule(P002, anc_rule)
)

P001 ANC: 1850 → MET
P002 ANC: 620 → NOT_MET


In [15]:
def evaluate_trial(patient, rules):
    results = []

    for rule in rules:
        if isinstance(rule["field"], list):
            patient_value = {
                field: get_nested_value(patient, field)
                for field in rule["field"]
            }
        else:
            patient_value = get_nested_value(
                patient,
                rule["field"]
            )

        status = evaluate_rule(
            patient,
            rule
        )

        results.append({
            "patient_id": patient["patient_id"],
            "rule_id": rule["rule_id"],
            "criterion_type": rule["criterion_type"],
            "field": rule["field"],
            "patient_value": patient_value,
            "operator": rule["operator"],
            "expected_value": rule["value"],
            "status": status,
            "description": rule["description"]
        })

    return results

In [16]:
def determine_overall_status(results):
    statuses = [
        result["status"]
        for result in results
    ]

    if "NOT_MET" in statuses:
        return "NOT_MET"

    if "UNKNOWN" in statuses:
        return "UNKNOWN"

    return "MET"

## Validate NCT07560826 With Synthetic Patients

P001 represents a patient expected to meet all encoded eligibility requirements.

P002 represents a patient expected not to meet the trial because of inadequate bone marrow and cardiac function.

In [17]:
P001_results = evaluate_trial(
    P001,
    rules_NCT07560826
)

P001_results_df = pd.DataFrame(P001_results)

P001_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "operator",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,operator,expected_value,status
0,NCT07560826_R01,age_years,15,<=,22,MET
1,NCT07560826_R02,diagnosis.cancer_type,osteosarcoma,==,osteosarcoma,MET
2,NCT07560826_R03,diagnosis.grade,high-grade,==,high-grade,MET
3,NCT07560826_R04,diagnosis.disease_extent,localized,in,"[localized, metastatic]",MET
4,NCT07560826_R05,anthropometrics.body_surface_area_m2,1.55,>=,0.8,MET
5,NCT07560826_R06,current_treatment.map_chemotherapy,True,==,True,MET
6,NCT07560826_R07,current_treatment.methotrexate,True,==,True,MET
7,NCT07560826_R08,current_treatment.methotrexate_dose_g_m2,12,==,12,MET
8,NCT07560826_R09,laboratory_values.bilirubin_uln_ratio,0.8,<=,1.5,MET
9,NCT07560826_R10,laboratory_values.alt_u_l,52,<=,135,MET


In [18]:
P002_results = evaluate_trial(
    P002,
    rules_NCT07560826
)

P002_results_df = pd.DataFrame(P002_results)

P002_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "operator",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,operator,expected_value,status
0,NCT07560826_R01,age_years,16,<=,22,MET
1,NCT07560826_R02,diagnosis.cancer_type,osteosarcoma,==,osteosarcoma,MET
2,NCT07560826_R03,diagnosis.grade,high-grade,==,high-grade,MET
3,NCT07560826_R04,diagnosis.disease_extent,metastatic,in,"[localized, metastatic]",MET
4,NCT07560826_R05,anthropometrics.body_surface_area_m2,1.48,>=,0.8,MET
5,NCT07560826_R06,current_treatment.map_chemotherapy,True,==,True,MET
6,NCT07560826_R07,current_treatment.methotrexate,True,==,True,MET
7,NCT07560826_R08,current_treatment.methotrexate_dose_g_m2,12,==,12,MET
8,NCT07560826_R09,laboratory_values.bilirubin_uln_ratio,1.1,<=,1.5,MET
9,NCT07560826_R10,laboratory_values.alt_u_l,68,<=,135,MET


In [19]:
print(
    "P001 overall:",
    determine_overall_status(P001_results)
)

print(
    "P002 overall:",
    determine_overall_status(P002_results)
)

P001 overall: MET
P002 overall: NOT_MET


In [20]:
P002_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,expected_value,status
0,NCT07560826_R01,age_years,16,22,MET
1,NCT07560826_R02,diagnosis.cancer_type,osteosarcoma,osteosarcoma,MET
2,NCT07560826_R03,diagnosis.grade,high-grade,high-grade,MET
3,NCT07560826_R04,diagnosis.disease_extent,metastatic,"[localized, metastatic]",MET
4,NCT07560826_R05,anthropometrics.body_surface_area_m2,1.48,0.8,MET
5,NCT07560826_R06,current_treatment.map_chemotherapy,True,True,MET
6,NCT07560826_R07,current_treatment.methotrexate,True,True,MET
7,NCT07560826_R08,current_treatment.methotrexate_dose_g_m2,12,12,MET
8,NCT07560826_R09,laboratory_values.bilirubin_uln_ratio,1.1,1.5,MET
9,NCT07560826_R10,laboratory_values.alt_u_l,68,135,MET


In [21]:
print("P001 status counts:")
print(P001_results_df["status"].value_counts())

print("\nP002 status counts:")
print(P002_results_df["status"].value_counts())

P001 status counts:
status
MET    18
Name: count, dtype: int64

P002 status counts:
status
MET        14
NOT_MET     4
Name: count, dtype: int64


## NCT07144254 — Tegavivint With Gemcitabine

The next trial evaluates patients with relapsed or refractory osteosarcoma. Its encoded rules include prior frontline therapy, performance status, treatment washout periods, organ function requirements, and relevant exclusions.

In [22]:
trial_07144254 = five_trials[
    five_trials["NCT Number"] == "NCT07144254"
].iloc[0]

print("Trial:")
print(trial_07144254["Title"])

print("\nEligibility Criteria:")
print(trial_07144254["Eligibility Text"])

Trial:
Tegavivint With Gemcitabine in Patients With Relapsed or Refractory Osteosarcoma

Eligibility Criteria:
Inclusion Criteria:

Diagnosis:

Participants must have had histologic verification of osteosarcoma at original diagnosis or relapse.

* All participants with relapsed or refractory osteosarcoma are eligible, provided they received front-line treatment with a regimen that contained at least 3 of the following agents: methotrexate, doxorubicin, cisplatin, and ifosfamide

  -Disease Status:
* Dose Escalation: Participants must have either measurable or evaluable disease per RECIST.Note: Participants with no evidence of disease on imaging (e.g., following pulmonary metastasectomy) are not eligible during the dose escalation phase.
* Dose Expansion: Participants with measurable or evaluable disease per RECIST and those with no evidence of disease on imaging following pulmonary metastasectomy are eligible during the dose expansion phase.

  -Performance Level: Participants must hav

## NCT07144254 — Tegavivint + Gemcitabine Symbolic Rules

These rules represent machine-checkable eligibility criteria for patients with relapsed or refractory osteosarcoma.

In [23]:
rules_NCT07144254 = [

    make_rule(
        "NCT07144254_R01",
        "diagnosis.histologic_confirmation",
        "==",
        True,
        "inclusion",
        "Osteosarcoma must have histologic verification."
    ),

    make_rule(
        "NCT07144254_R02",
        "diagnosis.disease_status",
        "in",
        ["relapsed", "refractory"],
        "inclusion",
        "Participant must have relapsed or refractory osteosarcoma."
    ),

    make_rule(
        "NCT07144254_R03",
        [
            "prior_treatment.methotrexate",
            "prior_treatment.doxorubicin",
            "prior_treatment.cisplatin",
            "prior_treatment.ifosfamide"
        ],
        "count_true_gte",
        3,
        "inclusion",
        "Front-line therapy must have included at least 3 of methotrexate, doxorubicin, cisplatin, and ifosfamide."
    ),

    make_rule(
        "NCT07144254_R04",
        "diagnosis.measurable_disease",
        "==",
        True,
        "inclusion",
        "Participant must have measurable or evaluable disease for the modeled cohort."
    ),

    make_rule(
        "NCT07144254_R05",
        "performance_status.score",
        ">=",
        60,
        "inclusion",
        "Lansky or Karnofsky performance score must be at least 60."
    ),

    make_rule(
        "NCT07144254_R06",
        "treatment_timing_days.last_chemotherapy",
        ">=",
        14,
        "inclusion",
        "At least 14 days must have passed since myelosuppressive chemotherapy."
    ),

    make_rule(
        "NCT07144254_R07",
        "treatment_timing_days.last_major_surgery",
        ">=",
        14,
        "inclusion",
        "At least 2 weeks must have passed since major surgery."
    ),

    make_rule(
        "NCT07144254_R08",
        "laboratory_values.anc_per_ul",
        ">=",
        750,
        "inclusion",
        "ANC must be at least 750/mm3."
    ),

    make_rule(
        "NCT07144254_R09",
        "laboratory_values.platelets_per_ul",
        ">=",
        75000,
        "inclusion",
        "Platelet count must be at least 75000/mm3."
    ),

    make_rule(
        "NCT07144254_R10",
        "laboratory_values.gfr_ml_min_1_73m2",
        ">=",
        70,
        "inclusion",
        "Creatinine clearance or GFR must be at least 70 mL/min/1.73m2."
    ),

    make_rule(
        "NCT07144254_R11",
        "laboratory_values.bilirubin_uln_ratio",
        "<=",
        1.5,
        "inclusion",
        "Bilirubin must be no more than 1.5 times ULN."
    ),

    make_rule(
        "NCT07144254_R12",
        "laboratory_values.alt_uln_ratio",
        "<=",
        5,
        "inclusion",
        "ALT must be no more than 5 times ULN."
    ),

    make_rule(
        "NCT07144254_R13",
        "pulmonary.dyspnea_at_rest",
        "==",
        False,
        "inclusion",
        "Participant must not have dyspnea at rest."
    ),

    make_rule(
        "NCT07144254_R14",
        "pulmonary.exercise_intolerance",
        "==",
        False,
        "inclusion",
        "Participant must not have exercise intolerance."
    ),

    make_rule(
        "NCT07144254_R15",
        "pulmonary.oxygen_requirement",
        "==",
        False,
        "inclusion",
        "Participant must not require supplemental oxygen."
    ),

    make_rule(
        "NCT07144254_R16",
        "pulmonary.oxygen_saturation_percent",
        ">",
        93,
        "inclusion",
        "Room-air oxygen saturation must be greater than 93 percent."
    ),

    make_rule(
        "NCT07144254_R17",
        "cardiac.qtc_ms",
        "<=",
        470,
        "inclusion",
        "QTc must be 470 ms or less."
    ),

    make_rule(
        "NCT07144254_R18",
        "other_clinical_facts.active_infection",
        "==",
        False,
        "exclusion",
        "Active uncontrolled infection is exclusionary."
    ),

    make_rule(
        "NCT07144254_R19",
        "other_clinical_facts.prior_tegavivint",
        "==",
        False,
        "exclusion",
        "Previous tegavivint treatment is not allowed."
    )
]

In [24]:
print(
    "Number of encoded rules:",
    len(rules_NCT07144254)
)

pd.DataFrame(rules_NCT07144254)

Number of encoded rules: 19


,rule_id,field,operator,value,criterion_type,description
0,NCT07144254_R01,diagnosis.histologic_confirmation,==,True,inclusion,Osteosarcoma must have histologic verification.
1,NCT07144254_R02,diagnosis.disease_status,in,"[relapsed, refractory]",inclusion,Participant must have relapsed or refractory o...
2,NCT07144254_R03,"[prior_treatment.methotrexate, prior_treatment...",count_true_gte,3,inclusion,Front-line therapy must have included at least...
3,NCT07144254_R04,diagnosis.measurable_disease,==,True,inclusion,Participant must have measurable or evaluable ...
4,NCT07144254_R05,performance_status.score,>=,60,inclusion,Lansky or Karnofsky performance score must be ...
5,NCT07144254_R06,treatment_timing_days.last_chemotherapy,>=,14,inclusion,At least 14 days must have passed since myelos...
6,NCT07144254_R07,treatment_timing_days.last_major_surgery,>=,14,inclusion,At least 2 weeks must have passed since major ...
7,NCT07144254_R08,laboratory_values.anc_per_ul,>=,750,inclusion,ANC must be at least 750/mm3.
8,NCT07144254_R09,laboratory_values.platelets_per_ul,>=,75000,inclusion,Platelet count must be at least 75000/mm3.
9,NCT07144254_R10,laboratory_values.gfr_ml_min_1_73m2,>=,70,inclusion,Creatinine clearance or GFR must be at least 7...


## Validate NCT07144254 With Synthetic Patients

P003 represents a strong match expected to meet every encoded requirement.

P004 represents a patient with several missing clinical values. Those missing facts should produce UNKNOWN results rather than incorrectly producing MET or NOT_MET.

In [25]:
P003_results = evaluate_trial(
    P003,
    rules_NCT07144254
)

P003_results_df = pd.DataFrame(P003_results)

P003_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,expected_value,status
0,NCT07144254_R01,diagnosis.histologic_confirmation,True,True,MET
1,NCT07144254_R02,diagnosis.disease_status,relapsed,"[relapsed, refractory]",MET
2,NCT07144254_R03,"[prior_treatment.methotrexate, prior_treatment...","{'prior_treatment.methotrexate': True, 'prior_...",3,MET
3,NCT07144254_R04,diagnosis.measurable_disease,True,True,MET
4,NCT07144254_R05,performance_status.score,80,60,MET
5,NCT07144254_R06,treatment_timing_days.last_chemotherapy,28,14,MET
6,NCT07144254_R07,treatment_timing_days.last_major_surgery,35,14,MET
7,NCT07144254_R08,laboratory_values.anc_per_ul,1420,750,MET
8,NCT07144254_R09,laboratory_values.platelets_per_ul,118000,75000,MET
9,NCT07144254_R10,laboratory_values.gfr_ml_min_1_73m2,88,70,MET


In [26]:
print(
    "P003 overall:",
    determine_overall_status(P003_results)
)

print(
    P003_results_df["status"].value_counts()
)

P003 overall: MET
status
MET    19
Name: count, dtype: int64


In [27]:
P004_results = evaluate_trial(
    P004,
    rules_NCT07144254
)

P004_results_df = pd.DataFrame(P004_results)

P004_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,expected_value,status
0,NCT07144254_R01,diagnosis.histologic_confirmation,True,True,MET
1,NCT07144254_R02,diagnosis.disease_status,refractory,"[relapsed, refractory]",MET
2,NCT07144254_R03,"[prior_treatment.methotrexate, prior_treatment...","{'prior_treatment.methotrexate': True, 'prior_...",3,MET
3,NCT07144254_R04,diagnosis.measurable_disease,True,True,MET
4,NCT07144254_R05,performance_status.score,70,60,MET
5,NCT07144254_R06,treatment_timing_days.last_chemotherapy,None,14,UNKNOWN
6,NCT07144254_R07,treatment_timing_days.last_major_surgery,28,14,MET
7,NCT07144254_R08,laboratory_values.anc_per_ul,1100,750,MET
8,NCT07144254_R09,laboratory_values.platelets_per_ul,92000,75000,MET
9,NCT07144254_R10,laboratory_values.gfr_ml_min_1_73m2,None,70,UNKNOWN


In [28]:
print(
    "P004 overall:",
    determine_overall_status(P004_results)
)

print(
    P004_results_df["status"].value_counts()
)

P004 overall: UNKNOWN
status
MET        15
UNKNOWN     4
Name: count, dtype: int64


## NCT05590572 — Sulfatinib for Relapsed/Refractory Osteosarcoma

Encode machine-checkable eligibility criteria and validate them using P005.

In [29]:
trial_05590572 = five_trials[
    five_trials["NCT Number"] == "NCT05590572"
].iloc[0]

print("Trial:")
print(trial_05590572["Title"])

print("\nEligibility Criteria:")
print(trial_05590572["Eligibility Text"])

Trial:
A Study of Sulfatinib on Relapsed or Refractory Drug Resistant Osteosarcoma

Eligibility Criteria:
Inclusion Criteria:

1. Osteosarcoma subjects Male or female participants aged 2 to 25 years at the time of informed consent(Histologically or cytologically confirmed diagnosis of high grade osteosarcoma)
2. Recurrent or refractory solid tumor malignancies that have treated with standard anticancer therapy but have no available treatment options.
3. Evaluable or measurable disease that met the following criteria: 1. Participants must have an evaluable or measurable disease based on RECIST 1.1, using computed tomography (CT)/ magnetic resonance imaging (MRI). 2. Lesions that have been treated locally, such as external beam radiation therapy (EBRT) or radiofrequency (RF) ablation, must subsequently grow clearly to be considered target lesions.
4. Life expectancy is 3 months or more.
5. Adequate bone marrow function : ①. Absolute neutrophil count (ANC) ≥ 1.0 x 10\^9/L. ②. Hemoglobin ≥

In [30]:
rules_NCT05590572 = [

    make_rule(
        "NCT05590572_R01",
        "age_years",
        ">=",
        2,
        "inclusion",
        "Participant must be at least 2 years old."
    ),

    make_rule(
        "NCT05590572_R02",
        "age_years",
        "<=",
        25,
        "inclusion",
        "Participant must be 25 years old or younger."
    ),

    make_rule(
        "NCT05590572_R03",
        "diagnosis.cancer_type",
        "==",
        "osteosarcoma",
        "inclusion",
        "Participant must have osteosarcoma."
    ),

    make_rule(
        "NCT05590572_R04",
        "diagnosis.histologic_confirmation",
        "==",
        True,
        "inclusion",
        "Diagnosis must be histologically or cytologically confirmed."
    ),

    make_rule(
        "NCT05590572_R05",
        "diagnosis.grade",
        "==",
        "high-grade",
        "inclusion",
        "Participant must have high-grade osteosarcoma."
    ),

    make_rule(
        "NCT05590572_R06",
        "diagnosis.disease_status",
        "in",
        ["relapsed", "recurrent", "refractory"],
        "inclusion",
        "Participant must have recurrent or refractory disease."
    ),

    make_rule(
        "NCT05590572_R07",
        "diagnosis.measurable_disease",
        "==",
        True,
        "inclusion",
        "Participant must have measurable or evaluable disease."
    ),

    make_rule(
        "NCT05590572_R08",
        "other_clinical_facts.life_expectancy_months",
        ">=",
        3,
        "inclusion",
        "Life expectancy must be at least 3 months."
    ),

    make_rule(
        "NCT05590572_R09",
        "laboratory_values.anc_per_ul",
        ">=",
        1000,
        "inclusion",
        "ANC must be at least 1000/uL."
    ),

    make_rule(
        "NCT05590572_R10",
        "laboratory_values.hemoglobin_g_dl",
        ">=",
        8.0,
        "inclusion",
        "Hemoglobin must be at least 8.0 g/dL."
    ),

    make_rule(
        "NCT05590572_R11",
        "laboratory_values.platelets_per_ul",
        ">=",
        75000,
        "inclusion",
        "Platelet count must be at least 75000/uL."
    ),

    make_rule(
        "NCT05590572_R12",
        "laboratory_values.bilirubin_uln_ratio",
        "<=",
        1.5,
        "inclusion",
        "Bilirubin must be no more than 1.5 times ULN."
    ),

    make_rule(
        "NCT05590572_R13",
        "laboratory_values.alt_uln_ratio",
        "<=",
        3,
        "inclusion",
        "ALT must be no more than 3 times ULN."
    ),

    make_rule(
        "NCT05590572_R14",
        "laboratory_values.ast_uln_ratio",
        "<=",
        3,
        "inclusion",
        "AST must be no more than 3 times ULN."
    ),

    make_rule(
        "NCT05590572_R15",
        "laboratory_values.alkaline_phosphatase_uln_ratio",
        "<=",
        3,
        "inclusion",
        "Alkaline phosphatase must be no more than 3 times ULN."
    ),

    make_rule(
        "NCT05590572_R16",
        "laboratory_values.gfr_ml_min_1_73m2",
        ">",
        70,
        "inclusion",
        "GFR or creatinine clearance must be greater than 70 mL/min/1.73m2."
    ),

    make_rule(
        "NCT05590572_R17",
        "cardiac.lvef_percent",
        ">=",
        50,
        "inclusion",
        "LVEF must be at least 50 percent."
    ),

    make_rule(
        "NCT05590572_R18",
        "other_clinical_facts.blood_pressure_controlled",
        "==",
        True,
        "inclusion",
        "Blood pressure must be adequately controlled."
    ),

    make_rule(
        "NCT05590572_R19",
        "other_clinical_facts.active_infection",
        "==",
        False,
        "exclusion",
        "Active infection is exclusionary."
    ),

    make_rule(
        "NCT05590572_R20",
        "other_clinical_facts.prior_sulfatinib",
        "==",
        False,
        "exclusion",
        "Previous sulfatinib treatment is not allowed."
    ),

    make_rule(
        "NCT05590572_R21",
        "other_clinical_facts.prior_vegf_vegfr_therapies_count",
        "<",
        2,
        "exclusion",
        "Two or more previous VEGF/VEGFR targeted therapies are exclusionary."
    ),

    make_rule(
        "NCT05590572_R22",
        "other_clinical_facts.pregnant",
        "==",
        False,
        "exclusion",
        "Pregnancy is exclusionary."
    ),

    make_rule(
        "NCT05590572_R23",
        "other_clinical_facts.breastfeeding",
        "==",
        False,
        "exclusion",
        "Breastfeeding is exclusionary."
    )
]

In [31]:
print("Number of encoded rules:", len(rules_NCT05590572))

pd.DataFrame(rules_NCT05590572)

Number of encoded rules: 23


,rule_id,field,operator,value,criterion_type,description
0,NCT05590572_R01,age_years,>=,2,inclusion,Participant must be at least 2 years old.
1,NCT05590572_R02,age_years,<=,25,inclusion,Participant must be 25 years old or younger.
2,NCT05590572_R03,diagnosis.cancer_type,==,osteosarcoma,inclusion,Participant must have osteosarcoma.
3,NCT05590572_R04,diagnosis.histologic_confirmation,==,True,inclusion,Diagnosis must be histologically or cytologica...
4,NCT05590572_R05,diagnosis.grade,==,high-grade,inclusion,Participant must have high-grade osteosarcoma.
5,NCT05590572_R06,diagnosis.disease_status,in,"[relapsed, recurrent, refractory]",inclusion,Participant must have recurrent or refractory ...
6,NCT05590572_R07,diagnosis.measurable_disease,==,True,inclusion,Participant must have measurable or evaluable ...
7,NCT05590572_R08,other_clinical_facts.life_expectancy_months,>=,3,inclusion,Life expectancy must be at least 3 months.
8,NCT05590572_R09,laboratory_values.anc_per_ul,>=,1000,inclusion,ANC must be at least 1000/uL.
9,NCT05590572_R10,laboratory_values.hemoglobin_g_dl,>=,8.0,inclusion,Hemoglobin must be at least 8.0 g/dL.


In [32]:
P005_results = evaluate_trial(
    P005,
    rules_NCT05590572
)

P005_results_df = pd.DataFrame(P005_results)

P005_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,expected_value,status
0,NCT05590572_R01,age_years,12,2,MET
1,NCT05590572_R02,age_years,12,25,MET
2,NCT05590572_R03,diagnosis.cancer_type,osteosarcoma,osteosarcoma,MET
3,NCT05590572_R04,diagnosis.histologic_confirmation,True,True,MET
4,NCT05590572_R05,diagnosis.grade,high-grade,high-grade,MET
5,NCT05590572_R06,diagnosis.disease_status,refractory,"[relapsed, recurrent, refractory]",MET
6,NCT05590572_R07,diagnosis.measurable_disease,True,True,MET
7,NCT05590572_R08,other_clinical_facts.life_expectancy_months,9,3,MET
8,NCT05590572_R09,laboratory_values.anc_per_ul,1700,1000,MET
9,NCT05590572_R10,laboratory_values.hemoglobin_g_dl,9.6,8.0,MET


In [33]:
print(
    "P005 overall:",
    determine_overall_status(P005_results)
)

print(
    P005_results_df["status"].value_counts()
)

P005 overall: MET
status
MET    23
Name: count, dtype: int64


In [34]:
trial_06235125 = five_trials[
    five_trials["NCT Number"] == "NCT06235125"
].iloc[0]

print("Trial:")
print(trial_06235125["Title"])

print("\nEligibility Criteria:")
print(trial_06235125["Eligibility Text"])

Trial:
Safety and Feasibility of Intraoperative Visualization With Cytalux in Children

Eligibility Criteria:
Inclusion Criteria:

1. Patients 6-17 years of age at the time of study enrollment
2. Willingness of research participant or legal guardian/representative to give written informed consent
3. Willingness of patients (subjects) age 12-17 to provide written adolescent assent
4. Patient weight greater than or equal to 20 kg
5. Histologically confirmed diagnosis of osteosarcoma, synovial sarcoma, hepatoblastoma, rhabdomyosarcoma, Ewing sarcoma, Wilms tumor or other non-rhabdomyosarcoma soft tissue sarcoma
6. Imaging findings highly suspicious for pulmonary metastatic disease based on CT, PET-CT or other imaging and warranting pulmonary surgery based on the judgment of the treating team. At least one nodule ≥4mm measured by preoperative imaging.
7. Female (assigned female at birth) participant is not pregnant and agrees to an acceptable form of contraception from the time of consent 

In [35]:
rules_NCT06235125 = [

    make_rule(
        "NCT06235125_R01",
        "age_years",
        ">=",
        6,
        "inclusion",
        "Participant must be at least 6 years old."
    ),

    make_rule(
        "NCT06235125_R02",
        "age_years",
        "<=",
        17,
        "inclusion",
        "Participant must be 17 years old or younger."
    ),

    make_rule(
        "NCT06235125_R03",
        "anthropometrics.weight_kg",
        ">=",
        20,
        "inclusion",
        "Participant must weigh at least 20 kg."
    ),

    make_rule(
        "NCT06235125_R04",
        "diagnosis.histologic_confirmation",
        "==",
        True,
        "inclusion",
        "Cancer diagnosis must be histologically confirmed."
    ),

    make_rule(
        "NCT06235125_R05",
        "diagnosis.cancer_type",
        "==",
        "osteosarcoma",
        "inclusion",
        "The modeled disease cohort is osteosarcoma."
    ),

    make_rule(
        "NCT06235125_R06",
        "surgical_imaging.pulmonary_metastatic_disease_suspected",
        "==",
        True,
        "inclusion",
        "Imaging must be highly suspicious for pulmonary metastatic disease."
    ),

    make_rule(
        "NCT06235125_R07",
        "surgical_imaging.pulmonary_surgery_recommended",
        "==",
        True,
        "inclusion",
        "Pulmonary surgery must be warranted based on the treating team's judgment."
    ),

    make_rule(
        "NCT06235125_R08",
        "surgical_imaging.largest_pulmonary_nodule_mm",
        ">=",
        4,
        "inclusion",
        "At least one pulmonary nodule must measure 4 mm or greater."
    ),

    make_rule(
        "NCT06235125_R09",
        "laboratory_values.gfr_ml_min_1_73m2",
        ">=",
        50,
        "inclusion",
        "eGFR must be at least 50 mL/min/1.73m2."
    ),

    make_rule(
        "NCT06235125_R10",
        "laboratory_values.alt_uln_ratio",
        "<=",
        3,
        "inclusion",
        "ALT must be no more than 3 times ULN."
    ),

    make_rule(
        "NCT06235125_R11",
        "laboratory_values.ast_uln_ratio",
        "<=",
        3,
        "inclusion",
        "AST must be no more than 3 times ULN."
    ),

    make_rule(
        "NCT06235125_R12",
        "laboratory_values.alkaline_phosphatase_uln_ratio",
        "<=",
        3,
        "inclusion",
        "Alkaline phosphatase must be no more than 3 times ULN."
    ),

    make_rule(
        "NCT06235125_R13",
        "laboratory_values.bilirubin_uln_ratio",
        "<=",
        2,
        "inclusion",
        "Total bilirubin must be no more than 2 times ULN."
    ),

    make_rule(
        "NCT06235125_R14",
        "other_clinical_facts.pregnant",
        "==",
        False,
        "exclusion",
        "Pregnancy is exclusionary."
    )
]

In [36]:
print("Number of encoded rules:", len(rules_NCT06235125))

pd.DataFrame(rules_NCT06235125)

Number of encoded rules: 14


,rule_id,field,operator,value,criterion_type,description
0,NCT06235125_R01,age_years,>=,6,inclusion,Participant must be at least 6 years old.
1,NCT06235125_R02,age_years,<=,17,inclusion,Participant must be 17 years old or younger.
2,NCT06235125_R03,anthropometrics.weight_kg,>=,20,inclusion,Participant must weigh at least 20 kg.
3,NCT06235125_R04,diagnosis.histologic_confirmation,==,True,inclusion,Cancer diagnosis must be histologically confir...
4,NCT06235125_R05,diagnosis.cancer_type,==,osteosarcoma,inclusion,The modeled disease cohort is osteosarcoma.
5,NCT06235125_R06,surgical_imaging.pulmonary_metastatic_disease_...,==,True,inclusion,Imaging must be highly suspicious for pulmonar...
6,NCT06235125_R07,surgical_imaging.pulmonary_surgery_recommended,==,True,inclusion,Pulmonary surgery must be warranted based on t...
7,NCT06235125_R08,surgical_imaging.largest_pulmonary_nodule_mm,>=,4,inclusion,At least one pulmonary nodule must measure 4 m...
8,NCT06235125_R09,laboratory_values.gfr_ml_min_1_73m2,>=,50,inclusion,eGFR must be at least 50 mL/min/1.73m2.
9,NCT06235125_R10,laboratory_values.alt_uln_ratio,<=,3,inclusion,ALT must be no more than 3 times ULN.


In [37]:
P006_results = evaluate_trial(
    P006,
    rules_NCT06235125
)

P006_results_df = pd.DataFrame(P006_results)

P006_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,expected_value,status
0,NCT06235125_R01,age_years,13,6,MET
1,NCT06235125_R02,age_years,13,17,MET
2,NCT06235125_R03,anthropometrics.weight_kg,46,20,MET
3,NCT06235125_R04,diagnosis.histologic_confirmation,True,True,MET
4,NCT06235125_R05,diagnosis.cancer_type,osteosarcoma,osteosarcoma,MET
5,NCT06235125_R06,surgical_imaging.pulmonary_metastatic_disease_...,True,True,MET
6,NCT06235125_R07,surgical_imaging.pulmonary_surgery_recommended,True,True,MET
7,NCT06235125_R08,surgical_imaging.largest_pulmonary_nodule_mm,8,4,MET
8,NCT06235125_R09,laboratory_values.gfr_ml_min_1_73m2,93,50,MET
9,NCT06235125_R10,laboratory_values.alt_uln_ratio,1.2,3,MET


In [38]:
print(
    "P006 overall:",
    determine_overall_status(P006_results)
)

print(
    P006_results_df["status"].value_counts()
)

P006 overall: MET
status
MET    14
Name: count, dtype: int64


In [39]:
trial_02013336 = five_trials[
    five_trials["NCT Number"] == "NCT02013336"
].iloc[0]

print("Trial:")
print(trial_02013336["Title"])

print("\nEligibility Criteria:")
print(trial_02013336["Eligibility Text"])

Trial:
Phase 1 Dose-escalating Study of MM-398 (Irinotecan Sucrosofate Liposome Injection) Plus Intravenous Cyclophosphamide in Recurrent or Refractory Pediatric Solid Tumors

Eligibility Criteria:
Inclusion Criteria:

* Histologically or cytologically-confirmed Ewing sarcoma, rhabdomyosarcoma, neuroblastoma, or osteosarcoma
* Disease progression after prior therapy in locally advanced or metastatic setting
* Measurable or evaluable disease based on the Response Evaluation Criteria in Solid Tumors (RECIST v1.1) criteria
* Age 12 months to \<21 years
* Adequate bone marrow reserves, hepatic function, and renal function
* Recovered from effects of any prior surgery or cancer therapy
* Patients 18 years or older will provide written consent. A parent or legal guardian of a patient \<18 years of age will provide informed consent and patients 11 to 18 years of age will provide written assent or as per participating institutional policy.

Exclusion Criteria:

* Clinically significant gastroi

In [40]:
rules_NCT02013336 = [

    make_rule(
        "NCT02013336_R01",
        "age_years",
        ">=",
        1,
        "inclusion",
        "Participant must be at least 12 months old."
    ),

    make_rule(
        "NCT02013336_R02",
        "age_years",
        "<",
        21,
        "inclusion",
        "Participant must be younger than 21 years."
    ),

    make_rule(
        "NCT02013336_R03",
        "diagnosis.cancer_type",
        "==",
        "osteosarcoma",
        "inclusion",
        "The modeled disease cohort is osteosarcoma."
    ),

    make_rule(
        "NCT02013336_R04",
        "diagnosis.histologic_confirmation",
        "==",
        True,
        "inclusion",
        "Cancer diagnosis must be histologically or cytologically confirmed."
    ),

    make_rule(
        "NCT02013336_R05",
        "diagnosis.disease_status",
        "in",
        ["recurrent", "relapsed", "refractory"],
        "inclusion",
        "The modeled patient must have recurrent, relapsed, or refractory disease after previous therapy."
    ),

    make_rule(
        "NCT02013336_R06",
        "diagnosis.disease_extent",
        "in",
        ["locally_advanced", "metastatic"],
        "inclusion",
        "Disease must have progressed in a locally advanced or metastatic setting."
    ),

    make_rule(
        "NCT02013336_R07",
        "diagnosis.measurable_disease",
        "==",
        True,
        "inclusion",
        "Participant must have measurable or evaluable disease."
    ),

    make_rule(
        "NCT02013336_R08",
        "other_clinical_facts.active_infection",
        "==",
        False,
        "exclusion",
        "Active infection is exclusionary."
    ),

    make_rule(
        "NCT02013336_R09",
        "other_clinical_facts.pregnant",
        "==",
        False,
        "exclusion",
        "Pregnancy is exclusionary."
    ),

    make_rule(
        "NCT02013336_R10",
        "other_clinical_facts.breastfeeding",
        "==",
        False,
        "exclusion",
        "Breastfeeding is exclusionary."
    )
]

In [41]:
print("Number of encoded rules:", len(rules_NCT02013336))

pd.DataFrame(rules_NCT02013336)

Number of encoded rules: 10


,rule_id,field,operator,value,criterion_type,description
0,NCT02013336_R01,age_years,>=,1,inclusion,Participant must be at least 12 months old.
1,NCT02013336_R02,age_years,<,21,inclusion,Participant must be younger than 21 years.
2,NCT02013336_R03,diagnosis.cancer_type,==,osteosarcoma,inclusion,The modeled disease cohort is osteosarcoma.
3,NCT02013336_R04,diagnosis.histologic_confirmation,==,True,inclusion,Cancer diagnosis must be histologically or cyt...
4,NCT02013336_R05,diagnosis.disease_status,in,"[recurrent, relapsed, refractory]",inclusion,"The modeled patient must have recurrent, relap..."
5,NCT02013336_R06,diagnosis.disease_extent,in,"[locally_advanced, metastatic]",inclusion,Disease must have progressed in a locally adva...
6,NCT02013336_R07,diagnosis.measurable_disease,==,True,inclusion,Participant must have measurable or evaluable ...
7,NCT02013336_R08,other_clinical_facts.active_infection,==,False,exclusion,Active infection is exclusionary.
8,NCT02013336_R09,other_clinical_facts.pregnant,==,False,exclusion,Pregnancy is exclusionary.
9,NCT02013336_R10,other_clinical_facts.breastfeeding,==,False,exclusion,Breastfeeding is exclusionary.


In [42]:
P007_results = evaluate_trial(
    P007,
    rules_NCT02013336
)

P007_results_df = pd.DataFrame(P007_results)

P007_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,expected_value,status
0,NCT02013336_R01,age_years,10,1,MET
1,NCT02013336_R02,age_years,10,21,MET
2,NCT02013336_R03,diagnosis.cancer_type,osteosarcoma,osteosarcoma,MET
3,NCT02013336_R04,diagnosis.histologic_confirmation,True,True,MET
4,NCT02013336_R05,diagnosis.disease_status,recurrent,"[recurrent, relapsed, refractory]",MET
5,NCT02013336_R06,diagnosis.disease_extent,metastatic,"[locally_advanced, metastatic]",MET
6,NCT02013336_R07,diagnosis.measurable_disease,True,True,MET
7,NCT02013336_R08,other_clinical_facts.active_infection,False,False,MET
8,NCT02013336_R09,other_clinical_facts.pregnant,False,False,MET
9,NCT02013336_R10,other_clinical_facts.breastfeeding,False,False,MET


In [43]:
print(
    "P007 overall:",
    determine_overall_status(P007_results)
)

print(
    P007_results_df["status"].value_counts()
)

P007 overall: MET
status
MET    10
Name: count, dtype: int64


In [44]:
all_trial_rules = {
    "NCT07560826": rules_NCT07560826,
    "NCT07144254": rules_NCT07144254,
    "NCT05590572": rules_NCT05590572,
    "NCT06235125": rules_NCT06235125,
    "NCT02013336": rules_NCT02013336
}

for nct_id, rules in all_trial_rules.items():
    print(nct_id, ":", len(rules), "rules")

NCT07560826 : 18 rules
NCT07144254 : 19 rules
NCT05590572 : 23 rules
NCT06235125 : 14 rules
NCT02013336 : 10 rules


In [45]:
showcase_cases = {
    "NCT07560826": P001,
    "NCT07144254": P003,
    "NCT05590572": P005,
    "NCT06235125": P006,
    "NCT02013336": P007
}

for nct_id, patient in showcase_cases.items():

    results = evaluate_trial(
        patient,
        all_trial_rules[nct_id]
    )

    overall = determine_overall_status(results)

    print(
        patient["patient_id"],
        "vs",
        nct_id,
        "→",
        overall
    )

P001 vs NCT07560826 → MET
P003 vs NCT07144254 → MET
P005 vs NCT05590572 → MET
P006 vs NCT06235125 → MET
P007 vs NCT02013336 → MET


## Full Patient-Trial Evaluation

Evaluate all 8 synthetic patients against all 5 symbolic trial rule sets.

In [46]:
all_evaluation_results = []

for patient in synthetic_patients:
    for nct_id, rules in all_trial_rules.items():

        rule_results = evaluate_trial(
            patient,
            rules
        )

        overall_status = determine_overall_status(
            rule_results
        )

        all_evaluation_results.append({
            "patient_id": patient["patient_id"],
            "nct_id": nct_id,
            "overall_status": overall_status,
            "met_count": sum(
                r["status"] == "MET"
                for r in rule_results
            ),
            "not_met_count": sum(
                r["status"] == "NOT_MET"
                for r in rule_results
            ),
            "unknown_count": sum(
                r["status"] == "UNKNOWN"
                for r in rule_results
            ),
            "rule_results": rule_results
        })

print(
    "Total patient-trial evaluations:",
    len(all_evaluation_results)
)

Total patient-trial evaluations: 40


In [47]:
evaluation_summary = []

for result in all_evaluation_results:
    evaluation_summary.append({
        "patient_id": result["patient_id"],
        "nct_id": result["nct_id"],
        "overall_status": result["overall_status"],
        "met_count": result["met_count"],
        "not_met_count": result["not_met_count"],
        "unknown_count": result["unknown_count"]
    })

evaluation_summary_df = pd.DataFrame(
    evaluation_summary
)

evaluation_summary_df.head(10)

,patient_id,nct_id,overall_status,met_count,not_met_count,unknown_count
0,P001,NCT07560826,MET,18,0,0
1,P001,NCT07144254,NOT_MET,7,2,10
2,P001,NCT05590572,NOT_MET,15,1,7
3,P001,NCT06235125,UNKNOWN,8,0,6
4,P001,NCT02013336,NOT_MET,8,2,0
5,P002,NCT07560826,NOT_MET,14,4,0
6,P002,NCT07144254,NOT_MET,5,4,10
7,P002,NCT05590572,NOT_MET,11,5,7
8,P002,NCT06235125,UNKNOWN,8,0,6
9,P002,NCT02013336,NOT_MET,9,1,0


In [48]:
eligibility_matrix = evaluation_summary_df.pivot(
    index="patient_id",
    columns="nct_id",
    values="overall_status"
)

eligibility_matrix

nct_id,NCT02013336,NCT05590572,NCT06235125,NCT07144254,NCT07560826
patient_id,,,,,
P001,NOT_MET,NOT_MET,UNKNOWN,NOT_MET,MET
P002,NOT_MET,NOT_MET,UNKNOWN,NOT_MET,NOT_MET
P003,UNKNOWN,UNKNOWN,UNKNOWN,MET,NOT_MET
P004,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,NOT_MET
P005,MET,MET,UNKNOWN,UNKNOWN,NOT_MET
P006,NOT_MET,NOT_MET,MET,NOT_MET,NOT_MET
P007,MET,UNKNOWN,UNKNOWN,NOT_MET,NOT_MET
P008,UNKNOWN,NOT_MET,NOT_MET,NOT_MET,NOT_MET


In [49]:
evaluation_summary_df[
    "overall_status"
].value_counts()

overall_status
NOT_MET    20
UNKNOWN    14
MET         6
Name: count, dtype: int64

In [50]:
unknown_cases = evaluation_summary_df[
    evaluation_summary_df["overall_status"] == "UNKNOWN"
]

unknown_cases

,patient_id,nct_id,overall_status,met_count,not_met_count,unknown_count
3,P001,NCT06235125,UNKNOWN,8,0,6
8,P002,NCT06235125,UNKNOWN,8,0,6
12,P003,NCT05590572,UNKNOWN,13,0,10
13,P003,NCT06235125,UNKNOWN,7,0,7
14,P003,NCT02013336,UNKNOWN,8,0,2
16,P004,NCT07144254,UNKNOWN,15,0,4
17,P004,NCT05590572,UNKNOWN,11,0,12
18,P004,NCT06235125,UNKNOWN,5,0,9
19,P004,NCT02013336,UNKNOWN,8,0,2
21,P005,NCT07144254,UNKNOWN,11,0,8


In [51]:
unknown_rule_details = []

for result in all_evaluation_results:

    if result["overall_status"] == "UNKNOWN":

        for rule_result in result["rule_results"]:

            if rule_result["status"] == "UNKNOWN":

                unknown_rule_details.append({
                    "patient_id": result["patient_id"],
                    "nct_id": result["nct_id"],
                    "rule_id": rule_result["rule_id"],
                    "field": rule_result["field"],
                    "description": rule_result["description"]
                })

unknown_rule_details_df = pd.DataFrame(
    unknown_rule_details
)

unknown_rule_details_df

,patient_id,nct_id,rule_id,field,description
0,P001,NCT06235125,NCT06235125_R06,surgical_imaging.pulmonary_metastatic_disease_...,Imaging must be highly suspicious for pulmonar...
1,P001,NCT06235125,NCT06235125_R07,surgical_imaging.pulmonary_surgery_recommended,Pulmonary surgery must be warranted based on t...
2,P001,NCT06235125,NCT06235125_R08,surgical_imaging.largest_pulmonary_nodule_mm,At least one pulmonary nodule must measure 4 m...
3,P001,NCT06235125,NCT06235125_R10,laboratory_values.alt_uln_ratio,ALT must be no more than 3 times ULN.
4,P001,NCT06235125,NCT06235125_R11,laboratory_values.ast_uln_ratio,AST must be no more than 3 times ULN.
...,...,...,...,...,...
79,P007,NCT06235125,NCT06235125_R08,surgical_imaging.largest_pulmonary_nodule_mm,At least one pulmonary nodule must measure 4 m...
80,P007,NCT06235125,NCT06235125_R11,laboratory_values.ast_uln_ratio,AST must be no more than 3 times ULN.
81,P007,NCT06235125,NCT06235125_R12,laboratory_values.alkaline_phosphatase_uln_ratio,Alkaline phosphatase must be no more than 3 ti...
82,P008,NCT02013336,NCT02013336_R09,other_clinical_facts.pregnant,Pregnancy is exclusionary.


In [52]:
def generate_missing_question(rule_result):

    field = rule_result["field"]

    field_questions = {
        "treatment_timing_days.last_chemotherapy":
            "When was the patient's most recent chemotherapy?",

        "treatment_timing_days.last_growth_factor":
            "When was the patient's most recent growth factor treatment?",

        "laboratory_values.gfr_ml_min_1_73m2":
            "What is the patient's current GFR or creatinine clearance?",

        "laboratory_values.alt_uln_ratio":
            "What is the patient's most recent ALT value relative to ULN?",

        "laboratory_values.ast_uln_ratio":
            "What is the patient's most recent AST value relative to ULN?",

        "laboratory_values.alkaline_phosphatase_uln_ratio":
            "What is the patient's most recent alkaline phosphatase value relative to ULN?",

        "cardiac.qtc_ms":
            "Is a recent QTc measurement available?",

        "cardiac.lvef_percent":
            "What is the patient's most recent LVEF?",

        "laboratory_values.anc_per_ul":
            "What is the patient's most recent ANC?",

        "laboratory_values.platelets_per_ul":
            "What is the patient's most recent platelet count?"
    }

    if isinstance(field, list):
        return (
            "Additional prior-treatment information is needed "
            "to evaluate this eligibility criterion."
        )

    return field_questions.get(
        field,
        f"Additional information is needed for: {field}"
    )

In [53]:
missing_questions = []

for result in all_evaluation_results:

    for rule_result in result["rule_results"]:

        if rule_result["status"] == "UNKNOWN":

            missing_questions.append({
                "patient_id": result["patient_id"],
                "nct_id": result["nct_id"],
                "rule_id": rule_result["rule_id"],
                "missing_field": rule_result["field"],
                "question": generate_missing_question(
                    rule_result
                )
            })

missing_questions_df = pd.DataFrame(
    missing_questions
)

missing_questions_df

,patient_id,nct_id,rule_id,missing_field,question
0,P001,NCT07144254,NCT07144254_R05,performance_status.score,Additional information is needed for: performa...
1,P001,NCT07144254,NCT07144254_R06,treatment_timing_days.last_chemotherapy,When was the patient's most recent chemotherapy?
2,P001,NCT07144254,NCT07144254_R07,treatment_timing_days.last_major_surgery,Additional information is needed for: treatmen...
3,P001,NCT07144254,NCT07144254_R12,laboratory_values.alt_uln_ratio,What is the patient's most recent ALT value re...
4,P001,NCT07144254,NCT07144254_R13,pulmonary.dyspnea_at_rest,Additional information is needed for: pulmonar...
...,...,...,...,...,...
184,P008,NCT06235125,NCT06235125_R11,laboratory_values.ast_uln_ratio,What is the patient's most recent AST value re...
185,P008,NCT06235125,NCT06235125_R12,laboratory_values.alkaline_phosphatase_uln_ratio,What is the patient's most recent alkaline pho...
186,P008,NCT06235125,NCT06235125_R14,other_clinical_facts.pregnant,Additional information is needed for: other_cl...
187,P008,NCT02013336,NCT02013336_R09,other_clinical_facts.pregnant,Additional information is needed for: other_cl...


In [54]:
matrix_output_path = (
    RULES_DIR / "patient_trial_eligibility_matrix.csv"
)

eligibility_matrix.to_csv(
    matrix_output_path
)

print(
    "Saved eligibility matrix to:",
    matrix_output_path.resolve()
)

Saved eligibility matrix to: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rules\patient_trial_eligibility_matrix.csv


In [55]:
summary_output_path = (
    RULES_DIR / "patient_trial_evaluation_summary.csv"
)

evaluation_summary_df.to_csv(
    summary_output_path,
    index=False
)

print(
    "Saved evaluation summary to:",
    summary_output_path.resolve()
)

Saved evaluation summary to: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rules\patient_trial_evaluation_summary.csv


In [56]:
detailed_output_path = (
    RULES_DIR / "patient_trial_rule_results.json"
)

with open(
    detailed_output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_evaluation_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved detailed rule results to:",
    detailed_output_path.resolve()
)

Saved detailed rule results to: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rules\patient_trial_rule_results.json


In [57]:
questions_output_path = (
    RULES_DIR / "missing_information_questions.csv"
)

missing_questions_df.to_csv(
    questions_output_path,
    index=False
)

print(
    "Saved missing-information questions to:",
    questions_output_path.resolve()
)

Saved missing-information questions to: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rules\missing_information_questions.csv


In [58]:
for path in [
    matrix_output_path,
    summary_output_path,
    detailed_output_path,
    questions_output_path
]:
    print(
        path.name,
        "→",
        path.exists()
    )

patient_trial_eligibility_matrix.csv → True
patient_trial_evaluation_summary.csv → True
patient_trial_rule_results.json → True
missing_information_questions.csv → True
